# COMP9517 Deep Learning Pipeline



## Setup

Imports the libraries used throughout — PyTorch and torchvision for the models, scikit-learn for the metrics, and pandas, NumPy and matplotlib for handling and plotting results — and selects the GPU if one is available.

In [ ]:
import os, re, glob, time, copy, json, pathlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models as tvm, transforms
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             balanced_accuracy_score)

warnings.filterwarnings('ignore', category=UserWarning)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150,
                     'axes.grid': True, 'grid.alpha': 0.3})

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Configuration

The three paths point at the project root, the dataset, and the output folder, and the two switches control whether a run is trained or simply scored from existing predictions. The RUNS dictionary holds all types of architecture, initialisation, augmentation setting, epoch count and learning rate used, and every results row later derives its labels from it.

In [ ]:
# ============ Colab setup — run once if  ============
from google.colab import drive
drive.mount('/content/drive')

# Unzip the team's prebuilt 500-class dataset from Drive to local disk (faster than reading off Drive).
# Comment out if data has already been built by Data Setup.ipy. Adjust the path as needed.
import subprocess
subprocess.run(['unzip', '-q', '-o', '/content/drive/MyDrive/COMP9517_Group_Project/COMP9517_Team_Subset.zip', '-d', '/content/'], check=True)

In [ ]:

ROOT     = Path(r'C:\Users\alexs\Desktop\UNSW\2026T2\COMP9517')
DATA_DIR = Path(r'C:\Users\alexs\Desktop\UNSW\2026T2\COMP9517\COMP9517_Team_500_40_10\content\Team_Dataset')
OUT_DIR  = ROOT / 'report_outputs'          # figures + summary.csv land here
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN    = False                            
RUN_ID   = 'convnext_tiny_Pre_Full_Aug'     
BEST_RUN = 'convnext_tiny_Pre_Full_Aug'     

SEED, NUM_CLASSES, BATCH_SIZE = 42, 500, 32
WEIGHT_DECAY, LABEL_SMOOTHING, IMAGE_SIZE = 1e-4, 0.1, 224
NUM_WORKERS = 8

# Keys must match your filenames: <RUN_ID>_predictions.csv, <RUN_ID>_history.csv
RUNS = {
 'r50_Scratch_No_Aug':         dict(arch='resnet50',      pretrained=False, augment='none',
                                    epochs=60, lr=1e-3, note='Baseline; control for augmentation'),
 'r50_Scratch_Full_Aug':       dict(arch='resnet50',      pretrained=False, augment='full',
                                    epochs=60, lr=1e-3, note='Augmentation effect'),
 'r50_Pre_Full_Aug':           dict(arch='resnet50',      pretrained=True,  augment='full',
                                    epochs=20, lr=1e-4, note='Transfer effect at 25.6M params'),
 'r18_Scratch_Full_Aug':       dict(arch='resnet18',      pretrained=False, augment='full',
                                    epochs=60, lr=1e-3, note='Capacity ablation from scratch'),
 'r18_Pre_Full_Aug':           dict(arch='resnet18',      pretrained=True,  augment='full',
                                    epochs=20, lr=1e-4, note='Transfer effect at 11.7M params'),
 'convnext_tiny_Pre_Full_Aug': dict(arch='convnext_tiny', pretrained=True,  augment='full',
                                    epochs=20, lr=1e-4, note='Modernised CNN, matched capacity'),
 'swin_t_Pre_Full_Aug':        dict(arch='swin_t',        pretrained=True,  augment='full',
                                    epochs=20, lr=1e-4, note='Transformer, matched capacity'),
}

SUMMARY_CSV = OUT_DIR / 'summary.csv'
print(f'{len(RUNS)} runs configured   ->  {OUT_DIR}')

## Helper Function

Functions used to image transform, constructs each architecture with a fresh 500-class head, load prediction files, re-rooting their stored image paths, and reading a species' genus from its label.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

EVAL_TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

TRAIN_TF = {
 'none': transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
 'full': transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.5, 1.0)),   # scale / pose variance
    transforms.RandomHorizontalFlip(),                            # orientation
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.03),                  # lighting
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2))]),        # occlusion
}

def build_model(arch, num_classes=NUM_CLASSES, pretrained=False):
    w = 'DEFAULT' if pretrained else None
    m = getattr(tvm, arch)(weights=w)
    if hasattr(m, 'fc'):                                   # resnet
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif hasattr(m, 'head'):                               # swin
        m.head = nn.Linear(m.head.in_features, num_classes)
    elif hasattr(m, 'classifier'):                         # convnext
        i = -1 if isinstance(m.classifier[-1], nn.Linear) else -2
        m.classifier[i] = nn.Linear(m.classifier[i].in_features, num_classes)
    return m

def n_params(m):  return sum(p.numel() for p in m.parameters() if p.requires_grad)
def short(c):     return ' '.join(c.split('_')[-2:])          # -> 'Juniperus deppeana'
def genus_of(c):  return c.split('_')[-2] if len(c.split('_')) >= 2 else c

def seed_everything(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def find_file(run_id, suffix):
    hits = list(ROOT.rglob(f'{run_id}_{suffix}'))
    return hits[0] if hits else None

def load_preds(run_id):
    #Read predictions.csv and re-root the stored filepaths at the local DATA_DIR.
    p = find_file(run_id, 'predictions.csv')
    if p is None:
        raise FileNotFoundError(f'No {run_id}_predictions.csv under {ROOT}')
    df = pd.read_csv(p)

    def remap(fp):
        parts = [q for q in str(fp).replace('\\', '/').split('/') if q]
        return str(DATA_DIR.joinpath(*parts[-3:])) if len(parts) >= 3 else str(fp)

    df['filepath'] = df['filepath'].map(remap)
    n_ok = df['filepath'].map(os.path.exists).sum()
    if n_ok < len(df):
        print(f'  ! {run_id}: {len(df)-n_ok}/{len(df)} image paths do not resolve '
              f'- check DATA_DIR (qualitative grids will show blanks)')
    return df

def load_hist(run_id):
    p = find_file(run_id, 'history.csv')
    return pd.read_csv(p) if p else None

def class_names_for(run_id=None):
    p = find_file(run_id, 'classes.txt') if run_id else None
    if p:  return p.read_text(encoding='utf-8').splitlines()
    return sorted(d.name for d in (DATA_DIR / 'test').iterdir() if d.is_dir())

def savefig(fig, name):
    path = OUT_DIR / name
    fig.tight_layout(); fig.savefig(path, bbox_inches='tight')
    print('  wrote', path.name); return path

print('helpers functions loaded')

## 0. Prediction Finder

Integrity Check: Scans the project folder for each run's prediction and history files and reports which runs are present.

In [ ]:
print(f"{'run':<28} {'arch':<14} {'init':<11} {'aug':<5} {'ep':>3} {'lr':>7}  {'preds':>6} {'hist':>5}")
print('-' * 92)
available = []
for rid, s in RUNS.items():
    has_p = find_file(rid, 'predictions.csv') is not None
    has_h = find_file(rid, 'history.csv') is not None
    if has_p: available.append(rid)
    print(f"{rid:<28} {s['arch']:<14} {'pretrained' if s['pretrained'] else 'scratch':<11} "
          f"{s['augment']:<5} {s['epochs']:>3} {s['lr']:>7.0e}  "
          f"{'yes' if has_p else '--':>6} {'yes' if has_h else '--':>5}")
print(f'\n{len(available)} run(s) with predictions found under {ROOT}')


## 1. Data Check

Ensures that the data used is up to specifications (500 classes, no repeated images for the validation and training sets, etc|)

In [ ]:
splits = {}
for s in ('train', 'val', 'test'):
    splits[s] = datasets.ImageFolder(DATA_DIR / s)

cn = splits['train'].classes
print(f'classes        : {len(cn)}')
for s, ds in splits.items():
    print(f'{s:<15}: {len(ds):>6} images  ({len(ds)//len(cn)} per class)')

assert splits['val'].classes == cn and splits['test'].classes == cn, 'class lists differ!'
print('\nclass lists identical across all three splits - no label shift')

files = {s: {Path(p).name for p, _ in ds.samples} for s, ds in splits.items()}
for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    n = len(files[a] & files[b])
    print(f'{a:<6} n {b:<5}: {n} shared filenames' + ('  <-- LEAKAGE' if n else '   ok'))

(OUT_DIR / 'class_list.txt').write_text('\n'.join(cn), encoding='utf-8')
print(f'\nclass_list.txt written ({len(cn)} species) - subset is reproducible')

## 2. Training

`train_run` trains one configuration, recording per-epoch loss, accuracy and wall-clock time, saving the history each epoch and keeping the checkpoint with the best validation accuracy; `predict` runs a trained model over the test set once and writes its predictions to a CSV file.

In [ ]:
def train_run(run_id, device=device):
    spec = RUNS[run_id]
    seed_everything()
    out = OUT_DIR / run_id; out.mkdir(parents=True, exist_ok=True)

    ds = {s: datasets.ImageFolder(DATA_DIR / s,
                                  TRAIN_TF[spec['augment']] if s == 'train' else EVAL_TF)
          for s in ('train', 'val')}
    dl = {s: DataLoader(d, batch_size=BATCH_SIZE, shuffle=(s == 'train'),
                        num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
                        pin_memory=torch.cuda.is_available())
          for s, d in ds.items()}
    sizes = {s: len(d) for s, d in ds.items()}

    model = build_model(spec['arch'], NUM_CLASSES, spec['pretrained']).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    opt   = optim.AdamW(model.parameters(), lr=spec['lr'], weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=spec['epochs'])

    print(f"=== {run_id} ===\n  {spec['arch']} | "
          f"{'ImageNet-1k' if spec['pretrained'] else 'random init'} | aug={spec['augment']} | "
          f"{spec['epochs']} ep | lr={spec['lr']:.0e} | {n_params(model)/1e6:.1f}M params")

    hist, best_wts, best_acc = [], copy.deepcopy(model.state_dict()), 0.0
    if device.type == 'cuda': torch.cuda.synchronize()
    t_run = time.time()

    for ep in range(1, spec['epochs'] + 1):
        t_ep = time.time(); row = {'epoch': ep, 'lr': opt.param_groups[0]['lr']}
        for phase in ('train', 'val'):
            model.train() if phase == 'train' else model.eval()
            loss_sum = correct = 0
            for x, y in dl[phase]:
                x, y = x.to(device), y.to(device)
                opt.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(phase == 'train'):
                    o = model(x); loss = crit(o, y)
                    if phase == 'train': loss.backward(); opt.step()
                loss_sum += loss.item() * x.size(0)
                correct  += (o.argmax(1) == y).sum().item()
            row[f'{phase}_loss'] = loss_sum / sizes[phase]
            row[f'{phase}_acc']  = correct / sizes[phase]

        if device.type == 'cuda': torch.cuda.synchronize()
        row['epoch_secs'] = time.time() - t_ep
        hist.append(row)

        if row['val_acc'] > best_acc:
            best_acc = row['val_acc']; best_wts = copy.deepcopy(model.state_dict())
            torch.save(best_wts, out / f'{run_id}_best.pth')

        print(f"  ep {ep:>3}/{spec['epochs']}  train {row['train_loss']:.4f}/{row['train_acc']:.4f}"
              f"  val {row['val_loss']:.4f}/{row['val_acc']:.4f}  ({row['epoch_secs']:.0f}s)")
        pd.DataFrame(hist).to_csv(out / f'{run_id}_history.csv', index=False)
        sched.step()

    train_secs = time.time() - t_run
    model.load_state_dict(best_wts)
    print(f'  done. best val {best_acc:.4f} in {train_secs/60:.1f} min')
    return dict(model=model, best_val_acc=best_acc, train_secs=train_secs,
                n_params=n_params(model))

print('train_run defined')

In [ ]:
def predict(run_id, model=None, split='test', device=device):
    """Write <run_id>_predictions.csv. This file is the interface for everything below -
    no model is loaded again after this point, so the whole results section can be
    regenerated on a laptop."""
    spec = RUNS[run_id]
    out = OUT_DIR / run_id; out.mkdir(parents=True, exist_ok=True)

    if model is None:
        ckpt = find_file(run_id, 'best.pth')
        if ckpt is None: raise FileNotFoundError(f'No {run_id}_best.pth under {ROOT}')
        model = build_model(spec['arch'], NUM_CLASSES, pretrained=False)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    model = model.to(device).eval()

    ds = datasets.ImageFolder(DATA_DIR / split, EVAL_TF)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time(); tops, confs = [], []
    with torch.no_grad():
        for x, _ in dl:
            p = model(x.to(device)).softmax(1)
            c, t = p.topk(5, dim=1)
            tops.append(t.cpu()); confs.append(c[:, 0].cpu())
    if device.type == 'cuda': torch.cuda.synchronize()
    infer_secs = time.time() - t0

    top5 = torch.cat(tops).numpy()
    df = pd.DataFrame({'filepath': [p for p, _ in ds.samples],
                       'true':     [y for _, y in ds.samples],
                       'pred':     top5[:, 0],
                       'confidence': torch.cat(confs).numpy()})
    for k in range(5): df[f'top{k+1}'] = top5[:, k]
    df.to_csv(out / f'{run_id}_predictions.csv', index=False)
    (out / f'{run_id}_classes.txt').write_text('\n'.join(ds.classes), encoding='utf-8')
    print(f'  {len(df)} images in {infer_secs:.1f}s ({infer_secs*1000/len(df):.2f} ms/image)')
    return df, infer_secs

# run only if TRAIN is on
if TRAIN:
    res = train_run(RUN_ID)
    df_new, infer_secs = predict(RUN_ID, model=res['model'])
else:
    print('TRAIN is False - using existing predictions. Set TRAIN=True to train a run.')

## 3. Performance Metrics

Computes the evaluation metrics from each run's predictions and assembles the results table: top-1 and top-5 accuracy, balanced accuracy, macro-averaged precision, recall and F1, and the number of species never once identified correctly.

In [ ]:
def compute_metrics(df, num_classes=NUM_CLASSES):
    """Works on any dataframe with true / pred / top1..top5 - so the traditional
    pipeline can be scored by this exact function and the comparison is genuinely
    like-for-like."""
    y, p = df['true'].values, df['pred'].values
    cols = [c for c in (f'top{k}' for k in range(1, 6)) if c in df.columns]
    top5 = float((df[cols].values == y[:, None]).any(1).mean()) if cols else np.nan
    mp, mr, mf, _   = precision_recall_fscore_support(y, p, average='macro',
                                                      labels=range(num_classes), zero_division=0)
    _, _, f1_per, _ = precision_recall_fscore_support(y, p, average=None,
                                                      labels=range(num_classes), zero_division=0)
    return dict(top1=float((p == y).mean()), top5=top5,
                balanced_acc=float(balanced_accuracy_score(y, p)),
                macro_p=float(mp), macro_r=float(mr), macro_f1=float(mf),
                species_never_correct=int((f1_per == 0).sum()), f1_per=f1_per)

def rescore_all(runs=None):
    rows = []
    for rid in (runs or RUNS):
        try: df = load_preds(rid)
        except FileNotFoundError: continue
        m = compute_metrics(df); s = RUNS[rid]
        h = load_hist(rid)
        best_val   = float(h['val_acc'].max()) if h is not None and 'val_acc' in h else np.nan
        train_secs = float(h['epoch_secs'].sum()) if h is not None and 'epoch_secs' in h else np.nan
        rows.append({'run': rid, 'arch': s['arch'], 'pretrained': s['pretrained'],
                     'augmentation': s['augment'], 'epochs': s['epochs'], 'lr': s['lr'],
                     'top1': m['top1'], 'top5': m['top5'], 'balanced_acc': m['balanced_acc'],
                     'macro_p': m['macro_p'], 'macro_r': m['macro_r'], 'macro_f1': m['macro_f1'],
                     'species_never_correct': m['species_never_correct'],
                     'best_val_acc': best_val, 'train_secs': train_secs,
                     'train_min': train_secs / 60 if train_secs == train_secs else np.nan,
                     'train_time_source': 'logged' if train_secs == train_secs else 'unlogged'})
    summary = pd.DataFrame(rows)
    order = {r: i for i, r in enumerate(RUNS)}
    summary = summary.sort_values('run', key=lambda c: c.map(order)).reset_index(drop=True)
    summary.to_csv(SUMMARY_CSV, index=False)
    return summary

summary = rescore_all()
print(f'{len(summary)} runs scored -> {SUMMARY_CSV}\n')
summary[['run', 'arch', 'pretrained', 'augmentation', 'top1', 'top5',
         'macro_f1', 'species_never_correct']]

## 3.1 Training Time Extrapolation

Adds a training-time column to the results table.

In [ ]:
def time_one_epoch(arch, augment='full', warmup=10):
    ds = datasets.ImageFolder(DATA_DIR / 'train', TRAIN_TF[augment])
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    m  = build_model(arch, NUM_CLASSES, False).to(device).train()
    o  = optim.AdamW(m.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
    c  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    it = iter(dl)
    for _ in range(warmup):                      # let cudnn.benchmark settle
        try: x, y = next(it)
        except StopIteration: break
        x, y = x.to(device), y.to(device)
        o.zero_grad(set_to_none=True); c(m(x), y).backward(); o.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        o.zero_grad(set_to_none=True); c(m(x), y).backward(); o.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    return time.time() - t0

SECS_PER_EPOCH = {a: time_one_epoch(a) for a in ('resnet18','resnet50','convnext_tiny','swin_t')}
print(SECS_PER_EPOCH)          

if SECS_PER_EPOCH:
    for i, r in summary.iterrows():
        if r['train_time_source'] == 'logged': continue
        if r['arch'] in SECS_PER_EPOCH:
            secs = SECS_PER_EPOCH[r['arch']] * RUNS[r['run']]['epochs']
            summary.loc[i, ['train_secs', 'train_min', 'train_time_source']] = \
                [secs, secs / 60, 'extrapolated']
    summary.to_csv(SUMMARY_CSV, index=False)
    print(summary[['run', 'arch', 'epochs', 'train_min', 'train_time_source']].to_string(index=False))
else:
    print('SECS_PER_EPOCH is empty - fill it in to populate the training-time column.')

## 3.2 Loss/Accuracy Curve

Plots training and validation loss and accuracy against epoch for every run, and prints the final and best validation accuracy alongside the train–validation gap.

In [ ]:
def plot_curves(run_ids=None):
    run_ids = run_ids or [r for r in RUNS if load_hist(r) is not None]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    for rid in run_ids:
        h = load_hist(rid)
        ln, = a1.plot(h.epoch, h.train_loss, label=rid)
        a1.plot(h.epoch, h.val_loss, '--', color=ln.get_color())
        a2.plot(h.epoch, h.train_acc, color=ln.get_color())
        a2.plot(h.epoch, h.val_acc, '--', color=ln.get_color())
    a1.set(xlabel='Epoch', ylabel='Loss', title='Loss (solid train, dashed val)')
    a2.set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy (solid train, dashed val)')
    a1.legend(fontsize=7)
    return savefig(fig, 'combined_curves.png')

plot_curves(); plt.show()

for rid in RUNS:
    h = load_hist(rid)
    if h is None: continue
    t = h.tail(3)
    print(f'{rid:<28} final train {t.train_acc.mean():.4f}  val {t.val_acc.mean():.4f}  '
          f'gap {t.train_acc.mean()-t.val_acc.mean():+.4f}  best val {h.val_acc.max():.4f}')

## 3.3 Confusion Analysis

Shows the full 500-class confusion matrix for overall structure, then an annotated zoom over the most-frequently-confused species, and lists the top confused pairs flagged by whether they share a genus.

In [ ]:
df_b = load_preds(BEST_RUN)
cn_b = class_names_for(BEST_RUN)
cm   = confusion_matrix(df_b['true'], df_b['pred'], labels=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='viridis', vmax=np.percentile(cm, 99.9))
fig.colorbar(im, ax=ax, label='count'); ax.grid(False)
ax.set(xlabel='Predicted', ylabel='True',
       title=f'Confusion matrix, {NUM_CLASSES} classes - {BEST_RUN}')
savefig(fig, f'{BEST_RUN}_confusion_full.png'); plt.show()

In [ ]:
off = cm.copy(); np.fill_diagonal(off, 0)
sym = off + off.T                       # mutual confusion strength
picked = []
for f in np.argsort(sym, axis=None)[::-1]:
    i, j = np.unravel_index(f, sym.shape)
    if sym[i, j] == 0: break
    for k in (i, j):
        if k not in picked: picked.append(int(k))
    if len(picked) >= 14: break
idx = picked[:14]

sub, labs = cm[np.ix_(idx, idx)], [short(cn_b[i]) for i in idx]
fig, ax = plt.subplots(figsize=(8.5, 7.5))
ax.imshow(sub, cmap='viridis'); ax.grid(False)
ax.set_xticks(range(len(idx)), labs, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(idx)), labs, fontsize=8)
for i in range(len(idx)):
    for j in range(len(idx)):
        if sub[i, j]:
            ax.text(j, i, sub[i, j], ha='center', va='center', fontsize=8,
                    color='white' if sub[i, j] < sub.max()*0.6 else 'black')
ax.set(xlabel='Predicted', ylabel='True', title=f'Most-confused species - {BEST_RUN}')
savefig(fig, f'{BEST_RUN}_confusion_zoom.png'); plt.show()

In [ ]:
# top pairs, flagged for shared genus
rows, cols = np.unravel_index(np.argsort(off, axis=None)[::-1][:15], off.shape)
pairs = pd.DataFrame([{'count': int(off[a, b]),
                       'true_species': short(cn_b[a]), 'pred_species': short(cn_b[b]),
                       'same_genus': genus_of(cn_b[a]) == genus_of(cn_b[b])}
                      for a, b in zip(rows, cols) if off[a, b] > 0])
print(f"{pairs['same_genus'].sum()} of the top {len(pairs)} confusions share a genus\n")

# what share of EACH model's errors are within-genus?
gs = []
for rid in RUNS:
    try: d = load_preds(rid)
    except FileNotFoundError: continue
    c = class_names_for(rid)
    e = d[d.pred != d.true]
    if not len(e): continue
    same = sum(genus_of(c[t]) == genus_of(c[p]) for t, p in zip(e['true'], e['pred']))
    gs.append({'run': rid, 'n_errors': len(e), 'same_genus': same, 'share': same/len(e)})
print(pd.DataFrame(gs).to_string(index=False), '\n')
pairs

## 3.4 Per-Class performance and Examples

Looks at performance species by species and at individual predictions. It plots the distribution of per-class F1 to show how evenly performance is spread across the 500 classes, then displays grids of confident correct predictions and of the most confident mistakes.

In [ ]:
m_b  = compute_metrics(df_b)
f1   = m_b['f1_per']
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(range(len(f1)), np.sort(f1)[::-1], width=1.0)
ax.set(xlabel='Species (sorted by F1)', ylabel='F1',
       title=f'Per-class F1 - {BEST_RUN}  ({int((f1==0).sum())} species never correct)')
savefig(fig, f'{BEST_RUN}_perclass_f1.png'); plt.show()

In [ ]:
def qual_grid(df, cn, kind='failure', n=8, seed=SEED):
    if kind == 'success':
        pool = df[(df.pred == df.true) & (df.confidence > 0.9)]
        rows = pool.sample(min(n, len(pool)), random_state=seed)
        title = f'Correct predictions - {BEST_RUN}'
    else:
        rows = df[df.pred != df.true].nlargest(n, 'confidence')
        title = f'Most confident errors - {BEST_RUN}'
    ncol = 4; nrow = int(np.ceil(len(rows) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3*ncol, 3.6*nrow))
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), rows.iterrows()):
        try: ax.imshow(Image.open(r.filepath).convert('RGB'))
        except Exception: ax.text(.5, .5, 'image not found', ha='center', va='center')
        ok = r.true == r.pred
        ax.set_title(f'T: {short(cn[r.true])}\nP: {short(cn[r.pred])} ({r.confidence:.2f})',
                     fontsize=8, color='green' if ok else 'red')
        ax.axis('off')
    for ax in np.atleast_1d(axes).ravel()[len(rows):]: ax.axis('off')
    fig.suptitle(title)
    return savefig(fig, f'{BEST_RUN}_qualitative_{kind}.png')

qual_grid(df_b, cn_b, 'success'); plt.show()
qual_grid(df_b, cn_b, 'failure'); plt.show()

## 3.5 Comparitive Summary

Prints the final results table across all runs and plots test accuracy against training time.

In [ ]:
view = summary[['run', 'arch', 'pretrained', 'augmentation', 'epochs',
                'top1', 'top5', 'macro_f1', 'train_min', 'species_never_correct']].copy()
for c in ('top1', 'top5', 'macro_f1'): view[c] = view[c].map('{:.4f}'.format)
view['train_min'] = view['train_min'].map(lambda v: f'{v:.1f}' if v == v else '-')
print(view.to_string(index=False))

In [ ]:
s = summary.dropna(subset=['train_min', 'top1'])
if len(s):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for _, r in s.iterrows():
        ax.scatter(r['train_min'], r['top1'], s=90,
                   marker='o' if r['pretrained'] else 's')
        ax.annotate(r['run'], (r['train_min'], r['top1']),
                    textcoords='offset points', xytext=(6, 4), fontsize=7)
    ax.set(xlabel='Training time (minutes)', ylabel='Test top-1 accuracy',
           title='Accuracy vs training cost  (circle pretrained, square scratch)')
    savefig(fig, 'accuracy_vs_time.png'); plt.show()
else:
    print('No training times yet - fill in SECS_PER_EPOCH in section 3.')